# Домашнє завдання: Рекомендаційні системи на реальних даних (Goodbooks-10k)

У цьому завданні Ви реалізуєте сучасні (advanced) архітектури рекомендаційних систем із фінального блоку лекції — але вже **не на іграшкових даних, а на реальному датасеті книжкових рейтингів Goodbooks-10k** (десятки тисяч користувачів, тисячі книг, мільйони оцінок).

Це дасть Вам змогу побачити, як підходи поводяться, коли даних справді багато: чому контентних ознак буває замало, як працює retrieval на тисячах елементів, і чому офлайн-метрики на кшталт Recall@K не такі високі, як хотілося б.

**Архітектури, які Ви зберете:** Vector Space Model, Two-Tower, Concat-based ranking (NCF) та двоетапний пайплайн Retrieval → Ranking.

**Стек:** `numpy`, `pandas`, `scikit-learn`, `torch`. GPU не обов'язковий, але з ним тренування буде швидшим (у Colab: *Runtime → Change runtime type → GPU*).

---

## Про датасет

[Goodbooks-10k](https://www.kaggle.com/datasets/zygmunt/goodbooks-10k) — це ~6 млн оцінок 10 000 найпопулярніших книг від 53 424 користувачів. Складається з кількох файлів:

- `ratings.csv` — оцінки: `user_id, book_id, rating` (1–5);
- `books.csv` — метадані книг: `book_id, goodreads_book_id, authors, title, average_rating, ...`;
- `book_tags.csv` — теги/полиці, які користувачі вішали на книги: `goodreads_book_id, tag_id, count`;
- `tags.csv` — розшифровка тегів: `tag_id, tag_name`.

**Важливий нюанс:** на відміну від навчального прикладу, тут **немає готових жанрів**. Жанри доведеться сконструювати самостійно з користувацьких тегів — а це шумні дані (юзери можуть зазначати що завгодно). Це реалістична задача feature engineering, і ми її розберемо в підготовчій частині.

Ще один нюанс із реальних даних: `book_tags.csv` посилається на `goodreads_book_id`, а `ratings.csv` — на `book_id`. Щоб їх поєднати, потрібен джойн через `books.csv`.


## Крок 0. Завантаження даних

Є три способи дістати дані — оберіть будь-який.

**Спосіб A — Kaggle API (рекомендований).** Завантаження з Kaggle API. Зручно, бо декілька файлів і вони завантажаться всі самостійно. Для цього способу завантажте свій `kaggle.json` (Kaggle → Account → Create New API Token), потім виконайте:
```python
from google.colab import files; files.upload()   # оберіть kaggle.json
```
і розкоментуйте відповідний блок нижче.

**Спосіб B — ручне завантаження.** Завантажте архів з посилання на датасет вище з Kaggle, розпакуйте і покладіть `ratings.csv`, `books.csv`, `book_tags.csv`, `tags.csv` поруч із ноутбуком (або через панель Files у Colab).

**Спосіб C — GitHub-дзеркало (фолбек).** Оригінальний автор виклав файли і на GitHub — код нижче підхопить їх автоматично, якщо локально файлів немає.


In [1]:
# (Спосіб A) Kaggle API — розкоментуйте, якщо завантажили kaggle.json
# !pip -q install kaggle
# import os, shutil
# os.makedirs("/root/.kaggle", exist_ok=True)
# shutil.move("kaggle.json", "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
# !kaggle datasets download -d zygmunt/goodbooks-10k --unzip -p .

In [2]:
import os
import numpy as np
import pandas as pd
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

GITHUB = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master"
DOWNLOADS_DIR = "/Users/myko/ML/ml_course/module_7_recommendation_systems/dataframes"
FILES = ["ratings.csv", "books.csv", "book_tags.csv", "tags.csv"]


def load(fname):
    local_path = os.path.join(DOWNLOADS_DIR, fname)
    """Спочатку шукаємо файл локально, інакше тягнемо з GitHub-дзеркала."""
    if os.path.exists(local_path):
        return pd.read_csv(local_path)
    print(f"{fname} не знайдено локально — завантажую з GitHub...")
    return pd.read_csv(f"{GITHUB}/{fname}")


ratings = load("ratings.csv")
books = load("books.csv")
book_tags = load("book_tags.csv")
tags = load("tags.csv")

print("ratings:", ratings.shape)
print("books:  ", books.shape)
print("book_tags:", book_tags.shape, "| tags:", tags.shape)
books[["book_id", "authors", "title", "average_rating"]].head()

ratings: (5976479, 3)
books:   (10000, 23)
book_tags: (999912, 3) | tags: (34252, 2)


,book_id,authors,title,average_rating
0,1,Suzanne Collins,"The Hunger Games (The Hunger Games, #1)",4.34
1,2,"J.K. Rowling, Mary GrandPré",Harry Potter and the Sorcerer's Stone (Harry P...,4.44
2,3,Stephenie Meyer,"Twilight (Twilight, #1)",3.57
3,4,Harper Lee,To Kill a Mockingbird,4.25
4,5,F. Scott Fitzgerald,The Great Gatsby,3.89


## Крок 1. Інженерія жанрів із тегів (feature engineering)

Жанрів у датасеті немає, але є користувацькі теги. Виберемо набір канонічних жанрів і для кожної книги позначимо, які з них їй приписали користувачі. Так ми отримаємо **бінарну матрицю book × genre** — це й будуть контентні ознаки айтемів (аналог `movie_feats_df` із лекції, але здобутий з реальних шумних даних).


In [3]:
# Канонічні жанри, які шукаємо серед тегів
GENRES = [
    "fantasy",
    "romance",
    "mystery",
    "thriller",
    "horror",
    "historical",
    "science-fiction",
    "young-adult",
    "nonfiction",
    "classics",
    "contemporary",
    "crime",
]

# tag_name -> tag_id
name_to_tagid = dict(zip(tags["tag_name"], tags["tag_id"]))
genre_tag_ids = {g: name_to_tagid[g] for g in GENRES if g in name_to_tagid}

# book_tags використовує goodreads_book_id -> мапимо у book_id через books.csv
gid_to_bid = dict(zip(books["goodreads_book_id"], books["book_id"]))
tagid_to_genre = {tid: g for g, tid in genre_tag_ids.items()}

bt = book_tags[book_tags["tag_id"].isin(genre_tag_ids.values())].copy()
bt["book_id"] = bt["goodreads_book_id"].map(gid_to_bid)
bt = bt.dropna(subset=["book_id"])
bt["genre"] = bt["tag_id"].map(tagid_to_genre)

# бінарна матриця book × genre (жанр присутній, якщо користувачі його тегали)
genre_matrix = (
    bt.pivot_table(
        index="book_id", columns="genre", values="count", aggfunc="sum", fill_value=0
    ).reindex(columns=GENRES, fill_value=0)
    > 0
).astype(int)

print(
    "Книг із хоча б одним жанром:",
    (genre_matrix.sum(axis=1) > 0).sum(),
    "/",
    len(books),
)
print("\nРозподіл жанрів:")
print(genre_matrix.sum().sort_values(ascending=False))
genre_matrix.head()

Книг із хоча б одним жанром: 9954 / 10000

Розподіл жанрів:
genre
contemporary       5287
fantasy            4259
romance            4251
mystery            3686
young-adult        3630
classics           2785
historical         2544
thriller           2522
science-fiction    2222
crime              2083
nonfiction         1833
horror             1372
dtype: int64


genre,fantasy,romance,mystery,thriller,horror,historical,science-fiction,young-adult,nonfiction,classics,contemporary,crime
book_id,,,,,,,,,,,,
1,1,1,0,1,0,0,1,1,0,0,1,0
2,1,0,1,0,0,0,0,1,0,1,1,0
3,1,0,0,0,1,0,1,1,0,0,1,0
4,0,0,1,0,0,1,0,1,0,1,1,1
5,0,1,0,0,0,1,0,1,0,1,0,0


## Крок 2. Підвибірка під Colab

6 млн рейтингів — забагато для навчального ноутбука на CPU. Візьмемо **топ-N найпопулярніших книг** і **активних користувачів** (хто поставив ≥ 20 оцінок), а тоді обмежимо число користувачів. Так зберігається щільність взаємодій, а тренування лишається швидким.

> Якщо у Вас GPU або багато часу — сміливо збільшуйте `TOP_BOOKS` та `N_USERS`.


In [4]:
TOP_BOOKS = 5000  # скільки найпопулярніших книг лишити
MIN_USER_RATINGS = 20  # мінімум оцінок на користувача
N_USERS = 15000  # скільки користувачів узяти у підвибірку
LIKE_THRESHOLD = 4  # rating >= 4 вважаємо "лайком" (позитивна взаємодія)

rng = np.random.RandomState(42)

top_books = ratings["book_id"].value_counts().head(TOP_BOOKS).index
r = ratings[ratings["book_id"].isin(top_books)]
active = r["user_id"].value_counts()
r = r[r["user_id"].isin(active[active >= MIN_USER_RATINGS].index)]
sample_users = rng.choice(
    r["user_id"].unique(), size=min(N_USERS, r["user_id"].nunique()), replace=False
)
r = r[r["user_id"].isin(sample_users)].copy()

# лишаємо тільки книги, для яких є жанрові ознаки
r = r[r["book_id"].isin(genre_matrix.index)].copy()

items = sorted(r["book_id"].unique())
users = sorted(r["user_id"].unique())
genre_matrix = genre_matrix.reindex(items).fillna(0).astype(int)

print(f"Взаємодій: {len(r):,} | користувачів: {len(users):,} | книг: {len(items):,}")
print(f"Щільність: {len(r) / (len(users) * len(items)):.4f}")

Взаємодій: 1,460,844 | користувачів: 15,000 | книг: 4,988
Щільність: 0.0195


In [5]:
import torch
import torch.nn as nn

torch.manual_seed(42)

user_to_idx = {u: i for i, u in enumerate(users)}
item_to_idx = {b: i for i, b in enumerate(items)}
title_of = dict(zip(books["book_id"], books["title"]))

item_feats = torch.tensor(genre_matrix.values, dtype=torch.float32).to(
    device
)  # (M, n_genres)
M = len(items)
n_genres = item_feats.shape[1]

# train/val split по взаємодіях
r = r.sample(frac=1, random_state=42).reset_index(drop=True)
n_val = int(len(r) * 0.2)
val_df = r.iloc[:n_val]
train_df = r.iloc[n_val:]

# позитивні пари (лайки) у train
train_pos = train_df[train_df["rating"] >= LIKE_THRESHOLD]
pos_u = torch.tensor(
    [user_to_idx[u] for u in train_pos["user_id"]], dtype=torch.int32
).to(device)
pos_i = torch.tensor(
    [item_to_idx[b] for b in train_pos["book_id"]], dtype=torch.int32
).to(device)

# що користувач уже бачив (щоб не рекомендувати повторно і не семплити як негатив)
from collections import defaultdict

seen_by_user = defaultdict(set)
for u, b in zip(train_df["user_id"], train_df["book_id"]):
    seen_by_user[user_to_idx[u]].add(item_to_idx[b])

# val-лайки для оцінки якості
val_pos = defaultdict(set)
for row in val_df.itertuples():
    if row.rating >= LIKE_THRESHOLD:
        val_pos[user_to_idx[row.user_id]].add(item_to_idx[row.book_id])

print(
    f"Позитивних пар у train: {len(pos_u):,} | користувачів з val-лайками: {len(val_pos):,}"
)

Позитивних пар у train: 805,998 | користувачів з val-лайками: 14,952


## Крок 3. Метрика оцінки якості рангування

В лекції ми з вами для оцінки якості використовували **RMSE**. Це валідний варіант, коли треба швидко оцінити якість рек. моделі, але спрощений. RMSE показує, наскільки точно модель передбачає оцінку, яку користувач поставить елементу.

В реальних системах нас ще цікавить **якість ранжування** — наскільки релевантні елементи потрапили в топ списку, який ми реально показуємо користувачу. Для цього використовують ранжувальні метрики: **Precision@K**, **Recall@K**, **NDCG**, **MAP**, **MRR**.

Детальніше можна познайомитись з цими мериками тут:
- огляд метрик для рекомендаційних систем: https://www.evidentlyai.com/ranking-metrics/evaluating-recommender-systems
- Precision та Recall at K: https://www.evidentlyai.com/ranking-metrics/precision-recall-at-k

Нижче давайте реалізуємо функцію `recall_at_k` і будемо оцінювати нею всі наші моделі.

![](https://cdn.prod.website-files.com/660ef16a9e0687d9cc27474a/662c4327f27ee08d3e4d4b2e_6577812c4d677925f1ab5f84_precision_recall_k9.png)

![](https://cdn.prod.website-files.com/660ef16a9e0687d9cc27474a/662c4327f27ee08d3e4d4b47_657781b1f9c868e0cda088f6_precision_recall_k11.png)

**Як працює `recall_at_k`:**

1. Для кожного користувача ми беремо його реальні вподобання з валідаційної вибірки (`val_pos` — книги, які він оцінив на ≥ 4), просимо модель оцінити всі книги й відбираємо топ-K рекомендацій. Перед цим прибираємо книги, які користувач уже бачив у train (щоб не рекомендувати відоме).

2. Далі рахуємо, скільки книг із топ-K справді потрапили в його вподобання (`hits`), і ділимо на загальну кількість релевантних книг (обмежену K, бо більше за K у топ і не влізе).

3. Усереднюємо по всіх користувачах — і отримуємо одне число від 0 до 1: **яку частку того, що користувачу реально сподобалось, модель змогла підняти в топ-K.**

In [6]:
def recall_at_k(score_fn, k=10):
    """Частка val-лайків, що потрапили у топ-k рекомендацій (усереднена по користувачах).
    score_fn(user_idx_tensor) -> матриця оцінок (n_users, M)."""
    eval_users = list(val_pos.keys())
    hits, total = 0, 0
    with torch.no_grad():
        scores = score_fn(torch.tensor(eval_users))  # (len(eval_users), M)
        for row, u in enumerate(eval_users):
            s = scores[row].clone()
            for i in seen_by_user[u]:
                s[i] = -1e9  # прибираємо вже побачене
            topk = torch.topk(s, k).indices.tolist()
            truth = val_pos[u]
            hits += len(set(topk) & truth)
            total += min(len(truth), k)
    return hits / max(total, 1)

---
## Завдання 1. Vector Space Model (векторний підхід)

Перетворимо і книги, і користувачів на вектори в спільному просторі та шукатимемо рекомендації через cosine similarity. Роль ембединга книги відіграє її **нормалізований вектор жанрів** (пояснення про нормалізацію - нижче), а вектор користувача збираємо як **average pooling** ембедингів книг, які він уподобав.

**Що зробити:**

1. Побудуйте `item_emb` — матрицю L2-нормалізованих жанрових векторів усіх книг.
2. Реалізуйте функцію `user_vector(user_idx)` — зважене (за оцінкою) середнє ембедингів уподобаних книг користувача.
3. Реалізуйте функцію `vsm_scores(user_idxs)` — оцінки (cosine) усіх книг для набору користувачів, та порахуйте `recall_at_k`.
4. Покажіть топ-5 рекомендацій для одного користувача (з назвами книг).

**Довідка:**

L2-нормалізація — це ділення вектора на його довжину (L2-норму), щоб отримати вектор тієї ж напрямленості, але одиничної довжини.

Норма рахується як корінь із суми квадратів компонент:

$$\|v\|_2 = \sqrt{(v_1^2 + v_2^2 + \dots + v_n^2)}$$

а сам нормалізований вектор — це
$$\hat{v} = \frac{v}{\|v\|_2}$$

Навіщо це в рекомендаційних системах: після нормалізації **косинусна подібність зводиться до простого скалярного добутку**. Бо $\cos(a, b) = \frac{a \cdot b}{\|a\|\|b\|}$, і якщо обидва вектори вже одиничної довжини, знаменник = 1, тож $\cos(a,b) = a \cdot b$. Це і швидше, і прибирає вплив «довжини» вектора — порівнюється лише напрямок (тобто склад жанрів/смаків), а не те, скільки книг користувач оцінив.

*Приклад:*

Вектор `[3, 4]` має довжину $\sqrt{(9+16)}=5$, після нормалізації стає `[0.6, 0.8]` — напрямок той самий, довжина 1.

In [7]:
norms = torch.norm(item_feats, p=2, dim=1, keepdim=True)
item_emb = item_feats / (norms + 1e-09)
num_users = len(user_to_idx)
user_emb_matrix = torch.zeros((num_users, n_genres), device=device)

pos_weights = torch.tensor(
    train_pos["rating"].values.astype(np.float32),
    dtype=torch.float32,
    device=device,
)

user_emb_matrix.index_add_(0, pos_u.long(), item_emb[pos_i] * pos_weights.unsqueeze(1))
user_weight_sum = torch.zeros(num_users, device=device)
user_weight_sum.index_add_(0, pos_u.long(), pos_weights)
user_weight_sum = torch.clamp(user_weight_sum, min=1.0)
user_emb_matrix = user_emb_matrix / user_weight_sum.unsqueeze(1)

user_emb_matrix = user_emb_matrix / (
    torch.norm(user_emb_matrix, p=2, dim=1, keepdim=True) + 1e-9
)


def user_vector(user_idxs):
    return user_emb_matrix[user_idxs.to(device)]


def vsm_scores(user_idxs):
    user_vecs = user_vector(user_idxs)
    return (user_vecs @ item_emb.T).cpu()


print("Recall@10:", recall_at_k(vsm_scores, k=10))

Recall@10: 0.034738087017842846


In [8]:
user_id_int = 0
user_idx = torch.tensor([user_id_int])
scores = vsm_scores(user_idx).squeeze().clone()
for item in seen_by_user[user_id_int]:
    scores[item] = -1e9
top_5 = torch.topk(scores, k=5).indices.tolist()
print("\nTop 5 recommendations:")
for i, idx in enumerate(top_5, 1):
    print(f"{i}. {title_of[items[idx]]}")


Top 5 recommendations:
1. Sophie's World
2. The Five People You Meet in Heaven
3. The Horse Whisperer
4. Oranges Are Not the Only Fruit
5. The Complete Persepolis


**Питання:** Recall@10 у векторного підходу досить низький. Чому?


Лише 12 жанрів, багато книг без жанрів або з шумними тегами.

---
## Завдання 2. Two-Tower архітектура

У Завданні 1 вектор користувача рахувався «вручну». Two-Tower натомість **навчає дві окремі башти**: User Tower (з ембединга user_id) та Item Tower (з жанрових ознак). Мережа зводить вектори уподобаних пар близько, а випадкових — далеко. Перевага: вектори книг рахуються один раз і кладуться в індекс (наприклад, FAISS) для швидкого retrieval — рахувати в реальному часі треба лише вектор користувача. Це **late fusion**.

**Що зробити:**

1. Реалізуйте `TwoTower` (user_tower через `nn.Embedding`, item_tower зі жанрових ознак), виходи L2-нормалізуйте.
2. Навчіть на лайках як позитивах і **negative sampling з усього корпусу** (як у пейпері від YouTube) з `BCEWithLogitsLoss` - він є реалізований в PyTorch.
3. Порахуйте `recall_at_k` через попередньо обчислені вектори книг і покажіть приклад рекомендацій.

> **Підказка.** Множте логіти на «температуру» (\~10), бо скалярний добуток нормалізованих векторів лежить у [-1, 1].
> Множення на температуру (\~10) розтягує діапазон логітів до [-10, 10], і тоді сигмоїда може видавати по-справжньому впевнені ймовірності (близькі до 0 і 1). Це дає лосу нормальний градієнт і модель навчається.


In [9]:
import torch.nn.functional as F

In [10]:
class TwoTower(nn.Module):
    def __init__(self, num_users, n_genres, emb_dim=32):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_net = nn.Linear(n_genres, emb_dim)
        self.temperature = 10.0

    def forward(self, user_idx, item_features):
        u_vecs = self.user_emb(user_idx)
        i_vecs = self.item_net(item_features)
        u_vecs = F.normalize(u_vecs, p=2, dim=1)
        i_vecs = F.normalize(i_vecs, p=2, dim=1)
        logits = (u_vecs * i_vecs).sum(dim=1) * self.temperature
        return logits

In [11]:
def get_batch(batch_size=256):
    idxs = torch.randint(len(pos_u), size=(batch_size,), device=device)
    batch_u_pos = pos_u[idxs]
    batch_i_pos = pos_i[idxs]
    batch_i_neg = torch.randint(0, M, size=(batch_size,), device=device)
    batch_u = torch.cat([batch_u_pos, batch_u_pos], dim=0)
    batch_i = torch.cat([batch_i_pos, batch_i_neg], dim=0)
    targets = torch.cat(
        [torch.ones(batch_size, device=device), torch.zeros(batch_size, device=device)],
        dim=0,
    )
    batch_feats = item_feats[batch_i]
    return batch_u, batch_feats, targets

In [12]:
model_two_tower = TwoTower(
    num_users=len(user_to_idx), n_genres=n_genres, emb_dim=32
).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model_two_tower.parameters(), lr=0.005)

for epoch in range(15001):
    model_two_tower.train()
    batch_u, batch_feats, targets = get_batch(1024)
    optimizer.zero_grad()
    logits = model_two_tower(batch_u, batch_feats)
    loss = criterion(logits, targets)
    loss.backward()
    optimizer.step()
    if epoch % 1000 == 0:
        print(f"Epoch {epoch}, loss: {loss:.3f}")

Epoch 0, loss: 0.987
Epoch 1000, loss: 0.656
Epoch 2000, loss: 0.573
Epoch 3000, loss: 0.556
Epoch 4000, loss: 0.541
Epoch 5000, loss: 0.532
Epoch 6000, loss: 0.531
Epoch 7000, loss: 0.538
Epoch 8000, loss: 0.537
Epoch 9000, loss: 0.529
Epoch 10000, loss: 0.545
Epoch 11000, loss: 0.515
Epoch 12000, loss: 0.505
Epoch 13000, loss: 0.529
Epoch 14000, loss: 0.522
Epoch 15000, loss: 0.507


In [13]:
def two_tower_scores(user_idxs):
    model_two_tower.eval()
    with torch.no_grad():
        u_vecs = model_two_tower.user_emb(user_idxs.to(device))
        i_vecs = model_two_tower.item_net(item_feats.to(device))
        u_vecs = F.normalize(u_vecs, p=2, dim=1)
        i_vecs = F.normalize(i_vecs, p=2, dim=1)
        scores = (u_vecs @ i_vecs.T) * model_two_tower.temperature
        return scores.cpu()


print("Two-Tower Recall@10:", recall_at_k(two_tower_scores, k=10))

Two-Tower Recall@10: 0.02669448260157346


In [14]:
user_id_int = 0
user_idx = torch.tensor([user_id_int])
scores = two_tower_scores(user_idx).squeeze().clone()
for item in seen_by_user[user_id_int]:
    scores[item] = -1e9
top_5 = torch.topk(scores, k=5).indices.tolist()
print("\nTop 5 recommendations:")
for i, idx in enumerate(top_5, 1):
    print(f"{i}. {title_of[items[idx]]}")


Top 5 recommendations:
1. The Complete Fairy Tales
2. The Virgin Suicides
3. The High King (The Chronicles of Prydain #5)
4. Esio Trot
5. Ella Enchanted


---
## Завдання 3. Concat-based ranking (NCF)

На відміну від Two-Tower (late fusion), тут **early fusion**: склеюємо ембединг користувача і ознаки книги в один вектор і пропускаємо через MLP, який сам моделює крос-взаємодії. Платою є те, що модель **не можна заіндексувати** — щоб знайти найкращу книгу, треба прогнати кожну пару (user, item). Тому її використовують лише на фінальному ранжуванні кількох кандидатів.

**Що зробити:**

1. Реалізуйте `NCF`: `concat(user_embedding, item_genre_features)` → MLP → один логіт.
2. Навчіть на тих самих позитивах/негативах.
3. Реалізуйте `rank_ncf(user_idx, candidate_idxs)` — ранжування заданого списку кандидатів за `sigmoid` логіта.


In [15]:
class NCF(nn.Module):
    def __init__(self, num_users, n_genres, emb_dim=32):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim + n_genres, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, user_idx, item_features):
        u_vec = self.user_emb(user_idx)
        x = torch.cat([u_vec, item_features], dim=1)
        logits = self.mlp(x)
        return logits.squeeze(1)


In [16]:
def rank_ncf(user_idx, candidate_idxs):
    model_nfc.eval()
    with torch.no_grad():
        item_ids = torch.tensor(candidate_idxs, dtype=torch.long, device=device)
        feats = item_feats[item_ids]
        num_candidates = len(candidate_idxs)
        user_ids = torch.full(
            (num_candidates,), user_idx, dtype=torch.long, device=device
        )
        logits = model_nfc(user_ids, feats)
        scores = torch.sigmoid(logits)
        return scores.cpu()

In [17]:
model_nfc = NCF(num_users=len(user_to_idx), n_genres=n_genres, emb_dim=32).to(device)
criterion_nfc = nn.BCEWithLogitsLoss()
optimizer_nfc = torch.optim.Adam(model_nfc.parameters(), lr=0.005)

for epoch in range(15001):
    model_nfc.train()
    batch_u, batch_feats, target = get_batch(1024)
    optimizer_nfc.zero_grad()
    logits = model_nfc(batch_u, batch_feats)
    loss = criterion_nfc(logits, target)
    loss.backward()
    optimizer_nfc.step()

    if epoch % 1000 == 0:
        print(f"Epoch {epoch}, loss: {loss.item():.3f}")


Epoch 0, loss: 0.695
Epoch 1000, loss: 0.551
Epoch 2000, loss: 0.532
Epoch 3000, loss: 0.502
Epoch 4000, loss: 0.514
Epoch 5000, loss: 0.501
Epoch 6000, loss: 0.494
Epoch 7000, loss: 0.484
Epoch 8000, loss: 0.472
Epoch 9000, loss: 0.479
Epoch 10000, loss: 0.468
Epoch 11000, loss: 0.483
Epoch 12000, loss: 0.477
Epoch 13000, loss: 0.459
Epoch 14000, loss: 0.432
Epoch 15000, loss: 0.463


In [18]:
def nfc_all_scores(user_idx):
    all_scores = []
    all_items = list(range(M))
    for u in user_idx:
        scores = rank_ncf(u.item(), all_items)
        all_scores.append(scores)
    return torch.stack(all_scores)


print("NCF Recall@10:", recall_at_k(nfc_all_scores, k=10))

NCF Recall@10: 0.04594771289841552


---
## Завдання 4. Двоетапний пайплайн Retrieval → Ranking

Поєднаємо все так, як це працює у великих системах: **Two-Tower швидко відбирає кандидатів** (retrieval серед усіх книг), а **NCF точно ранжує** цю коротку добірку.

**Що зробити:**

1. `retrieve(user_idx, n_candidates)` — топ-N книг за Two-Tower (Завдання 2), без уже побачених.
2. `recommend_pipeline(user_idx, n_candidates, top_k)` — прогнати кандидатів через `rank_ncf` (Завдання 3).
3. Показати для кількох користувачів: що відібрав retrieval і що залишив ranking.


In [19]:
def retrieve(user_idx_int, n_candidates=100):
    user_idx_tensor = torch.tensor([user_idx_int])
    scores = two_tower_scores(user_idx_tensor).squeeze().clone()
    for item in seen_by_user[user_idx_int]:
        scores[item] = -1e9
    top_candidates = torch.topk(scores, k=n_candidates).indices.tolist()
    return top_candidates


In [20]:
def recommend_pipeline(user_idx_int, n_candidates=100, top_k=10):
    candidates = retrieve(user_idx_int, n_candidates=n_candidates)
    nfc_scores = rank_ncf(user_idx_int, candidates)
    sorted_pairs = sorted(
        zip(candidates, nfc_scores.tolist()), key=lambda x: x[1], reverse=True
    )
    final_top_k = [item_id for item_id, _ in sorted_pairs[:top_k]]
    return candidates[:top_k], final_top_k

In [21]:
users_test = [10, 42, 88]
for user_id in users_test:
    retrieved_top, ranked_top = recommend_pipeline(user_id, n_candidates=50, top_k=10)

    print(f"\nUser results № {user_id}\n")
    print("Retrieved top Two-Tower:")
    for i, item_id in enumerate(retrieved_top, 1):
        print(f"{i}. Item {title_of[items[item_id]]}")

    print("\nRanked top NCF:")
    for i, item_id in enumerate(ranked_top, 1):
        print(f"{i}. Item {title_of[items[item_id]]}")


User results № 10

Retrieved top Two-Tower:
1. Item The Plague
2. Item Johnny Got His Gun
3. Item Go Set a Watchman
4. Item Angle of Repose
5. Item A Fine Balance
6. Item Regeneration (Regeneration, #1)
7. Item The Book of Ruth
8. Item The Known World
9. Item The Bonesetter's Daughter
10. Item The Stone Diaries

Ranked top NCF:
1. Item Rosencrantz and Guildenstern Are Dead
2. Item Paradise
3. Item Invisible Cities
4. Item The Penelopiad
5. Item The Amazing Adventures of Kavalier & Clay
6. Item Fool
7. Item Midnight's Children
8. Item Go Set a Watchman
9. Item Angle of Repose
10. Item A Fine Balance

User results № 42

Retrieved top Two-Tower:
1. Item The Princess Bride 
2. Item Angry Housewives Eating Bon Bons
3. Item Remarkable Creatures 
4. Item TransAtlantic
5. Item A Constellation of Vital Phenomena
6. Item The Birth House
7. Item Commonwealth
8. Item Breath, Eyes, Memory
9. Item The Buddha in the Attic
10. Item Last Night in Twisted River

Ranked top NCF:
1. Item The Princess Bri

**Питання:** навіщо ділити на два етапи, якщо можна ранжувати NCF одразу всі книги?

Two-Tower працює на основі швидкого матричного множення. Ми можемо заздалегідь порахувати ембединги для мільйонів книг, зберегти їх і миттєво відібрати ТОП-кандидатів за частки мілісекунди.

NCF — це важка нейромережа. Щоб оцінити книги, їй потрібно фізично склеїти ID користувача з кожною книгою окремо і прогнати ці пари через усі свої шари. Якщо спробувати прогнати так мільйон книг для кожного юзера в реальному часі, сервер впаде від навантаження.

---
## Завдання 5. Теоретичний блок (письмові відповіді)

Спираючись на лекцію та на те, що Ви щойно побачили на реальних даних, дайте розгорнуті відповіді в markdown-клітинці нижче.

1. **Чому Recall@10 такий низький?** На реальних даних усі моделі цього ДЗ дають скромний Recall@10. Назвіть щонайменше дві причини (підказки: бідні контентні ознаки — лише 12 жанрів; розрідженість; те, що val-лайки не охоплюють усіх книг, які користувач *міг би* вподобати).
2. **Як покращити якість, не змінюючи архітектуру?** Які додаткові ознаки книг і користувачів з Goodbooks можна було б під'єднати? (автор, рік, середній рейтинг, повний набір тегів через TF-IDF, текстові ембединги опису через BERT...)
3. **Diversity.** Якщо користувач любить фентезі, чому не варто показувати йому 10 фентезі-книг підряд? Як технічно підмішати різноманітність?
4. **Freshness / cold start.** Нова книга має 0 оцінок. Який підхід цього ДЗ зможе рекомендувати її одразу, а який — ні? Чому?
5. **Watch time > CTR (з лекції).** Поясніть, чому YouTube оптимізує час перегляду, а не CTR, і як це технічно вшито у weighted logistic regression.


1. Recall@10 у нашій роботі низький через кілька причин.

- Ми опираємось лише на 12 бінарних жанрових ознак, які дуже грубо описують книги. Багато книг отримують схожі або однакові жанрові профілі, і модель не може їх розрізняти.
- Дані дуже розріджені, і у валідації є небагато справжніх "лайків" для кожного користувача. Навіть якщо модель пропонує релевантні книги, вони можуть не збігатися з обмеженим набором валідаційних позитивів.
- Тегова інформація шумна: користувачі використовують різні позначення, а не чисті жанри. Це ще більше знижує якість content-based фіч.

2. Якість можна підвищити без зміни архітектури за рахунок додаткового feature engineering.

Для книг:
- автор і серія, бо користувачі часто читають декілька книг одного автора;
- рік публікації, середній рейтинг і кількість оцінок — це дає сигнал про популярність та актуальність;
- текстові ознаки: назва, опис, теги; їх можна закодувати через TF-IDF або предобучені BERT/CLIP ембедінги.

Для користувачів:
- жанровий профіль користувача, середня оцінка, частка лайків, кількість оцінених книг;
- історичні патерни поведінки: швидкість оцінювання, тренди зміни смаків;
- якщо доступні, сесійні ознаки (час, день тижня, тип взаємодії).

3. Якщо показувати 10 книг одного жанру підряд, користувач може втомитися і втратити інтерес. Різноманітність покращує досвід і дає шанс знайти релевантні, але інші за стилем варіанти.

Технічно це можна зробити так:
- додати penalty за схожість між уже відібраними рекомендаціями і новим кандидатом;
- використовувати MMR (Maximal Marginal Relevance);
- накладати обмеження на кількість книг одного жанру/автора у фінальному топі.

4. Freshness / cold start

Content-based підхід може рекомендувати нову книгу одразу, якщо для неї є метадані (жанри, автор, опис). У цьому ДЗ саме VSM і item-тower Two-Tower можуть працювати з новою книгою, бо item-вектор формується з контентних ознак.

Чистий collaborative filtering без ознак книги не зможе рекомендувати книгу з 0 оцінок, бо вона не має навченої ембединг-репрезентації і не входить у історичні взаємодії.

5. YouTube оптимізує watch time, а не CTR, тому що кліки самі по собі не гарантують утримання користувача. Відео можуть залучити кліком, але бути переглянутими лише кілька секунд.

Технічно це можна вшити у weighted logistic regression, де час перегляду або інша метрика engagement використовується як вага для кожного позитивного прикладу. Приклади з довгим watch time мають більший вклад у loss, і модель починає віддавати перевагу контенту, який реально утримує увагу, а не лише генерує кліки.